# Understanding Joins with Food Data

## What is a Join?

Imagine you have two lists:
- **List A**: Everything in your kitchen (inventory)
- **List B**: Everything a recipe needs (ingredients)

A "join" is matching items between these two lists. Different types of joins answer different questions:
- **Inner Join**: "What ingredients do I have that a recipe also needs?" (only matches)
- **Left Join**: "For everything in my kitchen, which recipes use it?" (all inventory, matched where possible)
- **Right Join**: "For everything a recipe needs, do I have it?" (all ingredients, matched where possible)
- **Outer Join**: "Show me everything from both lists" (complete picture)

This is one of the most important concepts in data analysis — once you understand joins, you can connect any two datasets.

## Setup: Load and Prepare Data

Before we can practice joins, we need two DataFrames that share a common column (a "key") to join on. We'll create:
1. **ingredients_df** — a flat table of recipe ingredients (one row per ingredient per recipe)
2. **inv_for_join** — a deduplicated table of active (non-expired) inventory items

Both tables share a `join_key` column in the format `"category:item_name"`.

In [ ]:
import sys
import json
from pathlib import Path

# Add the project root to Python's module search path
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.database import get_engine

engine = get_engine()

# Load recipes and inventory
recipes_df = pd.read_sql("SELECT * FROM recipe", engine)
inventory_df = pd.read_sql("SELECT * FROM activeinventory", engine)

# Parse the JSON ingredients column
recipes_df["parsed_ingredients"] = recipes_df["ingredients"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Create a flat ingredients DataFrame — one row per ingredient per recipe
# This "explodes" the nested JSON into rows suitable for joining
rows = []
for _, recipe in recipes_df.iterrows():
    for ing in recipe["parsed_ingredients"]:
        if isinstance(ing, dict):
            rows.append({
                "recipe_name": recipe["name"],
                "ingredient_name": ing.get("name", ""),
                "ingredient_category": ing.get("category", "other"),
                "quantity": ing.get("quantity", ""),
                "unit": ing.get("unit", ""),
                # Create a join key matching the inventory format: "category:name"
                "join_key": f"{ing.get('category', 'other')}:{ing.get('name', '')}",
            })

ingredients_df = pd.DataFrame(rows)

# Prepare a deduplicated inventory for joining (one row per unique item)
inv_for_join = (
    inventory_df[inventory_df["is_expired"] == False]
    .groupby("join_key")
    .first()
    .reset_index()[["join_key", "item_name", "category", "quantity", "unit"]]
    .rename(columns={
        "item_name": "inv_item_name",
        "category": "inv_category",
        "quantity": "inv_quantity",
        "unit": "inv_unit",
    })
)

print(f"Recipe ingredients (flat): {len(ingredients_df)} rows")
print(f"Active inventory items: {len(inv_for_join)} rows")
print(f"\nSample ingredients:")
print(ingredients_df.head())
print(f"\nSample inventory:")
print(inv_for_join.head())

## Section 1: Inner Join — What Matches in Both?

An inner join keeps ONLY rows that have a match in **both** tables. If an ingredient isn't in your inventory, it's dropped. If an inventory item isn't in any recipe, it's also dropped.

**Question answered**: "Which ingredients do I have that recipes also need?"

```python
pd.merge(left, right, on="key", how="inner")
```

In [ ]:
# pd.merge() combines two DataFrames based on a shared column (the "key")
# how="inner" means: only keep rows where the key exists in BOTH DataFrames

inner_result = pd.merge(
    ingredients_df,      # Left table: what recipes need
    inv_for_join,        # Right table: what you have
    on="join_key",       # The column to match on
    how="inner",         # Only keep matches
)

print(f"Inner join result: {len(inner_result)} rows")
print(f"(These are ingredients that appear in BOTH recipes AND inventory)\n")

# Show a clean view
display_cols = ["recipe_name", "ingredient_name", "inv_item_name", "inv_quantity", "inv_unit"]
print(inner_result[display_cols].head(15).to_string(index=False))

In [ ]:
# Which inventory items are used by the MOST recipes?
# value_counts() on the matched ingredient names tells us
item_usage = inner_result["ingredient_name"].value_counts()
print("Inventory items used by the most recipes:")
print(item_usage.head(10))

## Section 2: Left Join — Start from Inventory

A left join keeps ALL rows from the **left** table (inventory), and matches from the right table (recipes) where possible. If an inventory item isn't used by any recipe, it still appears — but with `NaN` in the recipe columns.

**NaN** (Not a Number) is pandas' way of saying "no data here." In a left join, NaN means "no match was found in the right table."

**Question answered**: "For each item I own, which recipes use it? And what don't I use?"

In [ ]:
# how="left" means: keep ALL rows from the LEFT table (inv_for_join)
# If no match exists in ingredients_df, recipe columns will be NaN

left_result = pd.merge(
    inv_for_join,         # Left: start from what you HAVE
    ingredients_df,       # Right: match to what recipes NEED
    on="join_key",
    how="left",
)

print(f"Left join result: {len(left_result)} rows")
print(f"Inventory items with NO recipe match:")

# Filter to rows where recipe_name is NaN — these items aren't in any recipe
no_recipe = left_result[left_result["recipe_name"].isna()]
print(f"  {len(no_recipe)} inventory items aren't used by any recipe:\n")
for _, row in no_recipe.iterrows():
    print(f"  - {row['inv_item_name']} ({row['inv_category']})")

## Section 3: Right Join — Start from Recipes (THE KEY JOIN)

This is **the most important join** in our application. A right join keeps ALL rows from the **right** table (recipe ingredients), and matches from the left table (inventory) where possible.

**This is how recipe matching works!** For each ingredient a recipe needs:
- Match found → you have it ✓
- NaN (no match) → you're missing it ✗

**Question answered**: "For each ingredient a recipe needs, do I have it?"

In [ ]:
# how="right" means: keep ALL rows from the RIGHT table (ingredients_df)
# NaN in the inventory columns means "you DON'T have this ingredient"

right_result = pd.merge(
    inv_for_join,         # Left: what you HAVE
    ingredients_df,       # Right: what recipes NEED (keep ALL of these)
    on="join_key",
    how="right",
)

print(f"Right join result: {len(right_result)} rows")

# Add a status column for clarity
right_result["status"] = right_result["inv_item_name"].apply(
    lambda x: "✓ Have it" if pd.notna(x) else "✗ MISSING"
)

# Show results for a specific recipe
sample_recipe = right_result["recipe_name"].value_counts().index[0]
recipe_detail = right_result[right_result["recipe_name"] == sample_recipe]

print(f"\nIngredient check for '{sample_recipe}':")
for _, row in recipe_detail.iterrows():
    print(f"  {row['status']:12s}  {row['ingredient_name']}")

# Summary across all recipes
total = len(right_result)
have = right_result["inv_item_name"].notna().sum()
missing = right_result["inv_item_name"].isna().sum()
print(f"\nOverall: {have}/{total} ingredients available ({have/total*100:.0f}%)")
print(f"Missing: {missing} ingredients across all recipes")

In [ ]:
# The power of the right join: find EXACTLY what's missing
# Filter to rows where inv_item_name is NaN (no inventory match)

missing_ingredients = right_result[right_result["inv_item_name"].isna()]

print("Missing ingredients (needed by recipes but not in inventory):")
missing_counts = missing_ingredients["ingredient_name"].value_counts()
for ingredient, count in missing_counts.head(15).items():
    print(f"  {ingredient}: needed by {count} recipe(s)")

## Section 4: Outer (Full) Join — The Complete Picture

An outer join keeps ALL rows from **both** tables. It's the union of left join and right join. You get:
- Items only in inventory (NaN in recipe columns)
- Items only in recipes (NaN in inventory columns)
- Items in both (all columns filled)

**Question answered**: "Show me absolutely everything — all my food AND all recipe ingredients."

In [ ]:
# how="outer" means: keep ALL rows from BOTH tables
# This gives the most complete (but also largest) result

outer_result = pd.merge(
    inv_for_join,
    ingredients_df,
    on="join_key",
    how="outer",
)

print(f"Outer join result: {len(outer_result)} rows")

# Categorize each row
def categorize_match(row):
    has_inv = pd.notna(row["inv_item_name"])
    has_recipe = pd.notna(row["recipe_name"])
    if has_inv and has_recipe:
        return "In BOTH"
    elif has_inv:
        return "Inventory ONLY (no recipe uses it)"
    else:
        return "Recipe ONLY (not in inventory)"

outer_result["match_type"] = outer_result.apply(categorize_match, axis=1)

# Count each category
print("\nMatch breakdown:")
print(outer_result["match_type"].value_counts())

## Join Type Summary

| Join Type | Keeps from Left (Inventory) | Keeps from Right (Recipes) | Use When |
|-----------|---------------------------|---------------------------|----------|
| **Inner** | Only matched | Only matched | "What overlaps?" |
| **Left** | ALL | Only matched | "What in my inventory is used?" |
| **Right** | Only matched | ALL | "Can I make this recipe?" |
| **Outer** | ALL | ALL | "Show me everything" |

**Key insight**: The RIGHT JOIN is the foundation of recipe matching — it starts from what recipes need and checks if you have it. A `NaN` in the inventory column means "missing ingredient."

## Exercises

1. **Modify the right join** to only show rows where the inventory match is NaN (missing ingredients). Which recipe has the most missing ingredients?

2. **Use the inner join** result to find which of your inventory items are used by the most recipes. What does this tell you about staple ingredients?

3. **Challenge**: Create a "recipe scorecard" — for each recipe, calculate what percentage of its ingredients you have. Which recipe are you closest to being able to make?

*Hint for exercise 3*: After a right join, group by `recipe_name` and calculate `notna().mean()` on the inventory column.